# DNU EDA — Comparación de clasificadores

Natalia Debandi

Análisis exploratorio del dataset DNU 70/2017 extendido con las predicciones de seis clasificadores de discurso de odio.

| Clasificador | Tipo | Etiqueta positiva |
|---|---|---|
| **HATEFUL** | Manual contextual (piuba-bigdata/beto) | OR de todas las categorías excepto `CALLS` |
| **GPT** | GPT-4o zero-shot (Batch API) | `gpt_is_xenophobic = True` |
| **Cardiff** | XLM-RoBERTa (CardiffNLP) | `HATE` |
| **BNE** | RoBERTa (BNE, hate+offensive) | `HATE` o `OFFENSIVE` |
| **BETO** | BETO fine-tuned hate speech v2 | `LABEL_1` |
| **pysentimiento** | RoBERTuito (pysentimiento) | lista no vacía de categorías ⚠️ solo 2021-03-05 |

## 1. Imports y configuración

In [ ]:
import re
import html as htmllib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

FILE_V2   = 'data/DNU_clasificadores_v2.csv'
FILE_BASE = 'data/tweets_dnu_clasificados.csv'

CLASIF = ['HATEFUL', 'GPT', 'Cardiff', 'BNE', 'BETO', 'pysentimiento']
LABEL_COLS = ['WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL', 'CALLS']

COLORS = {
    'HATEFUL':       '#2c3e50',
    'GPT':           '#2980b9',
    'Cardiff':       '#27ae60',
    'BNE':           '#e74c3c',
    'BETO':          '#8e44ad',
    'pysentimiento': '#e67e22',
}

## 2. Carga de datos

In [ ]:
# Dataset principal con predicciones de todos los modelos
df = pd.read_csv(FILE_V2, low_memory=False)
df['fecha']     = pd.to_datetime(df['fecha'], utc=True)
df['fecha_dia'] = pd.to_datetime(df['fecha_dia'])

# Columnas adicionales del dataset base (etiquetas manuales individuales + reply info)
extra = pd.read_csv(
    FILE_BASE,
    usecols=['id', 'in_reply_to_status_id', 'n_labels'] + LABEL_COLS,
    low_memory=False,
)
df = df.merge(extra, on='id', how='left')

# Tipo de tweet detectado por texto
def _tipo(text):
    t = str(text)
    if t.startswith('RT @'):  return 'RT'
    elif t.startswith('@'):   return 'Mención/Reply'
    else:                     return 'Original'
df['tipo'] = df['text'].apply(_tipo)

# Usuario del tweet original (solo en RTs)
df['rt_user'] = df['text'].str.extract(r'^RT @(\w+):', expand=False)

print(f'Dataset: {df.shape[0]:,} tweets × {df.shape[1]} columnas')
print(f'Tipos: {df["tipo"].value_counts().to_dict()}')
df[['id', 'text', 'fecha_dia', 'tipo'] + CLASIF].head(3)

## 3. Cobertura temporal

In [ ]:
por_dia = (
    df.groupby('fecha_dia').size()
    .reset_index(name='tweets')
    .sort_values('fecha_dia')
)
por_dia['acumulado']    = por_dia['tweets'].cumsum()
por_dia['% del total']  = (por_dia['tweets'] / por_dia['tweets'].sum() * 100).round(1)

print(f"Período: {por_dia['fecha_dia'].min().date()} → {por_dia['fecha_dia'].max().date()}")
print(f"Total:   {por_dia['tweets'].sum():,} tweets")

display(
    por_dia
    .assign(fecha_dia=por_dia['fecha_dia'].dt.strftime('%d %b %Y'))
    .set_index('fecha_dia')
    .style
    .format({'tweets': '{:,}', 'acumulado': '{:,}', '% del total': '{:.1f}%'})
    .background_gradient(subset=['tweets'], cmap='Blues')
    .set_caption('Tweets recolectados por día')
)

## 4. Cobertura y tasa de odio por clasificador

In [ ]:
n = len(df)
rows = []
for m in CLASIF:
    cov  = int(df[m].notna().sum())
    hate = int((df[m] == 1).sum())
    rows.append({
        'Clasificador':       m,
        'Tweets cubiertos':   cov,
        '% cobertura':        cov / n * 100,
        'Tweets odiosos':     hate,
        '% odio (cubiertos)': hate / cov * 100 if cov else 0,
    })

cov_df = pd.DataFrame(rows).set_index('Clasificador')
display(
    cov_df.style
    .format({'Tweets cubiertos': '{:,}', 'Tweets odiosos': '{:,}',
             '% cobertura': '{:.1f}%', '% odio (cubiertos)': '{:.1f}%'})
    .background_gradient(subset=['% odio (cubiertos)'], cmap='Reds')
    .background_gradient(subset=['Tweets cubiertos'],   cmap='Blues')
    .set_caption('Cobertura y tasa de odio por clasificador')
)

## 5. Serie temporal de odio por clasificador

Porcentaje de tweets clasificados como odiosos por día para cada modelo.
pysentimiento (\*) solo cubre 2021-03-05.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for m in CLASIF:
    daily = (
        df[df[m].notna()]
        .groupby('fecha_dia')
        .agg(n=('id', 'count'), n_hate=(m, 'sum'))
    )
    rate  = daily['n_hate'] / daily['n'] * 100
    label = f'{m} *' if m == 'pysentimiento' else m
    ax.plot(rate.index, rate.values,
            marker='o', linewidth=2.2, markersize=6,
            label=label, color=COLORS[m])
    # Etiqueta en el último punto
    ax.annotate(f'{rate.iloc[-1]:.1f}%',
                xy=(rate.index[-1], rate.iloc[-1]),
                xytext=(5, 0), textcoords='offset points',
                fontsize=9, color=COLORS[m], va='center')

ax.set_ylabel('% tweets odiosos', fontsize=11)
ax.set_title('Tasa de odio diaria por clasificador', fontsize=13)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%d %b'))
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/EDA_clasif_serie_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

### 5b. Heatmap: tasa de odio por día × clasificador

In [ ]:
heat_rows = {}
for m in CLASIF:
    daily = df[df[m].notna()].groupby('fecha_dia').agg(n=('id','count'), n_hate=(m,'sum'))
    heat_rows[m] = (daily['n_hate'] / daily['n'] * 100).round(1)

heat_df = pd.DataFrame(heat_rows).T
heat_df.columns = heat_df.columns.strftime('%d %b')

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(
    heat_df, annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.5, linecolor='#eee', ax=ax,
    cbar_kws={'label': '% odio', 'shrink': 0.8},
)
ax.set_title('Tasa de odio (%) por clasificador y día', fontsize=12, pad=10)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('outputs/EDA_clasif_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Distribución por categoría manual

Conteo de tweets con cada etiqueta individual del clasificador contextual (piuba-bigdata/beto).

In [ ]:
counts = df[LABEL_COLS].astype(float).sum().sort_values(ascending=False)
pcts   = (counts / len(df) * 100).round(2)

label_df = pd.DataFrame({'tweets': counts.astype(int), '% del total': pcts})
label_df.index.name = 'etiqueta'

print(f"Tweets con ≥1 etiqueta (incluyendo CALLS): {int((df['n_labels'] > 0).sum()):,}")
print(f"Tweets HATEFUL (sin CALLS):                {int((df['HATEFUL'] == 1).sum()):,}")

display(
    label_df.style
    .format({'tweets': '{:,}', '% del total': '{:.2f}%'})
    .background_gradient(subset=['tweets'], cmap='Reds')
    .set_caption('Tweets por categoría manual de discurso de odio')
)

# Gráfico
fig, ax = plt.subplots(figsize=(10, 4))
colors  = ['#ef4444' if l != 'CALLS' else '#ea580c' for l in counts.index]
ax.bar(counts.index, counts.values, color=colors, width=0.6)
for i, (lbl, val) in enumerate(counts.items()):
    ax.text(i, val + 30, f'{int(val):,}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Tweets')
ax.set_title('Frecuencia por categoría manual')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))
plt.tight_layout()
plt.savefig('outputs/EDA_clasif_labels.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Tipos de tweet por clasificador

Tasa de odio según el tipo de tweet (RT, Mención/Reply, Original) para cada clasificador.

In [ ]:
# Tabla: % odio por tipo × clasificador
tipo_counts = df.groupby('tipo').size().rename('total')
tipo_hate   = df.groupby('tipo')[CLASIF].apply(lambda g: (g == 1).mean() * 100).round(1)
tipo_df     = tipo_hate.join(tipo_counts)
tipo_df.insert(0, 'total', tipo_df.pop('total'))

display(
    tipo_df.style
    .format({'total': '{:,}', **{m: '{:.1f}%' for m in CLASIF}})
    .background_gradient(subset=CLASIF, cmap='Reds', axis=None)
    .set_caption('% tweets odiosos por tipo de tweet y clasificador')
)

# Gráfico de barras agrupadas
tipos  = tipo_df.index.tolist()
x      = np.arange(len(tipos))
n_mod  = len(CLASIF)
w      = 0.12

fig, ax = plt.subplots(figsize=(11, 5))
for i, m in enumerate(CLASIF):
    offset = (i - n_mod / 2 + 0.5) * w
    bars = ax.bar(x + offset, tipo_df[m], width=w, label=m, color=COLORS[m], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(tipos, fontsize=11)
ax.set_ylabel('% tweets odiosos')
ax.set_title('Tasa de odio por tipo de tweet y clasificador')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(ncol=3, fontsize=9)
plt.tight_layout()
plt.savefig('outputs/EDA_clasif_tipos.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Top RTs más retuiteados

Tweets más retuiteados en el dataset, con la etiqueta de cada clasificador.

In [ ]:
TOP_N = 15

top_rts = (
    df[df['tipo'] == 'RT']
    .groupby('text')
    .agg(
        n_rts      = ('id',        'count'),
        fecha_min  = ('fecha_dia', 'min'),
        fecha_max  = ('fecha_dia', 'max'),
        **{m: (m, 'first') for m in CLASIF},
    )
    .reset_index()
    .sort_values('n_rts', ascending=False)
    .head(TOP_N)
    .reset_index(drop=True)
)

def _model_badges(row):
    parts = []
    for m in CLASIF:
        try:
            if pd.notna(row[m]) and int(row[m]) == 1:
                parts.append(
                    f'<span style="background:{COLORS[m]};color:#fff;padding:2px 7px;'
                    f'border-radius:10px;font-size:11px;margin-right:3px;">{m}</span>'
                )
        except Exception:
            pass
    return ''.join(parts) or '<span style="color:#aaa;font-size:11px;">sin etiqueta</span>'

cards = []
for i, row in top_rts.iterrows():
    texto  = htmllib.escape(str(row['text'])[:200])
    span   = int((row['fecha_max'] - row['fecha_min']).days)
    fechas = f"{row['fecha_min'].strftime('%d %b')} → {row['fecha_max'].strftime('%d %b')}" if span > 0 else row['fecha_min'].strftime('%d %b')
    badges = _model_badges(row)
    cards.append(
        f'<div style="background:#fff;border:1px solid #dde;border-left:4px solid #e74c3c;'
        f'border-radius:8px;padding:10px 14px;margin:6px 0;">'
        f'  <div style="display:flex;justify-content:space-between;margin-bottom:6px;">'
        f'    <span style="font-weight:bold;color:#222;font-size:13px;">#{i+1} — {row["n_rts"]:,} RTs</span>'
        f'    <span style="font-size:11px;color:#888;">📅 {fechas}</span>'
        f'  </div>'
        f'  <div style="background:#fff;font-size:13px;color:#222;line-height:1.5;margin-bottom:7px;">{texto}…</div>'
        f'  <div style="background:#fff;">{badges}</div>'
        f'</div>'
    )

display(HTML(
    '<h4 style="font-family:Arial;color:#222;">Top RTs más retuiteados</h4>' +
    ''.join(cards)
))

## 9. Usuarios más retuiteados

Para cada usuario extraído de los RTs (`RT @usuario:`): cantidad de RTs en el dataset y tasa de odio según cada clasificador.

In [ ]:
TOP_USERS = 30

user_stats = (
    df[df['rt_user'].notna()]
    .groupby('rt_user')
    .agg(
        n_rts = ('id', 'count'),
        **{m: (m, lambda x: (x == 1).mean() * 100) for m in CLASIF},
    )
    .reset_index()
    .sort_values('n_rts', ascending=False)
    .head(TOP_USERS)
)

display(
    user_stats.set_index('rt_user')
    .style
    .format({'n_rts': '{:,.0f}', **{m: '{:.1f}%' for m in CLASIF}})
    .background_gradient(subset=['n_rts'],   cmap='Blues')
    .background_gradient(subset=['HATEFUL'], cmap='Reds')
    .background_gradient(subset=['BNE'],     cmap='Oranges')
    .set_caption(f'Top {TOP_USERS} usuarios más retuiteados — tasa de odio por clasificador')
)

## 10. Exploración por keyword

Busca tweets que contengan alguno de los keywords y estén etiquetados como odiosos según el filtro manual seleccionado.
Muestra tarjetas individuales con fondo blanco y badges por modelo.

In [ ]:
# ── Configuración ─────────────────────────────────────────────────────────────
SEED_KW     = 42
N_TWEETS_KW = 10
KEYWORDS    = ['DNU', 'vacuna']   # lista OR
FILTRO_KW   = 'HATEFUL'           # 'HATEFUL', 'RACISM', o None

# ── Filtrar ───────────────────────────────────────────────────────────────────
pattern = '|'.join(KEYWORDS)
mask_kw = df['text'].str.contains(pattern, case=False, na=False, regex=True)
mask_hate = df[FILTRO_KW] == 1 if FILTRO_KW else pd.Series(True, index=df.index)

filtered  = df[mask_kw & mask_hate].drop_duplicates(subset=['text'])

print(f"Tweets con ({pattern}):       {mask_kw.sum():,}")
print(f"  con {FILTRO_KW}=1 (dedup): {len(filtered):,}")

if len(filtered) == 0:
    print('Sin resultados.')
else:
    muestra = filtered.sample(min(N_TWEETS_KW, len(filtered)), random_state=SEED_KW)

    output_html = (
        f'<h4 style="font-family:Arial;color:#222;margin-bottom:12px;">'
        f'Tweets odiosos con "{pattern}" — SEED={SEED_KW} '
        f'<span style="font-size:12px;color:#888;">({len(filtered):,} únicos)</span></h4>'
    )

    for _, row in muestra.iterrows():
        text   = htmllib.escape(str(row['text'])[:400])
        date   = str(row.get('fecha_dia', ''))[:10]
        badges = _model_badges(row)
        tipo   = '🔁' if str(row['text']).startswith('RT @') else '🐦'
        output_html += (
            f'<div style="font-family:Arial;background:#fff;border:1px solid #ccd;'
            f'border-radius:10px;margin-bottom:10px;overflow:hidden;">'
            f'  <div style="background:#f4f6f9;padding:5px 14px;font-size:11px;color:#666;'
            f'border-bottom:1px solid #dde;">{tipo} {date}</div>'
            f'  <div style="background:#fff;padding:10px 14px;font-size:13px;color:#222;'
            f'line-height:1.6;">{text}</div>'
            f'  <div style="background:#fff;padding:5px 14px 10px;">{badges}</div>'
            f'</div>'
        )

    display(HTML(output_html))